# HDBSCAN Clustering Demonstration

This notebook demonstrates the HDBSCAN (Hierarchical DBSCAN) clustering algorithm implemented in Rust using various scikit-learn datasets.

## Overview

HDBSCAN extends DBSCAN by:
- Building a hierarchy of clusters at different density levels
- Automatically extracting the most stable clusters
- Handling clusters of varying densities
- Providing cluster membership probabilities
- Eliminating the need to choose the `eps` parameter

**Parameters:**
- `min_cluster_size`: The minimum number of samples in a cluster
- `min_samples`: The number of samples in a neighborhood for core distance calculation (optional)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from ghdbscan import HDBSCAN
import time

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Helper Functions

In [ ]:
def plot_clusters(X, labels, title, ax=None):
    """Plot clustering results."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))
    
    unique_labels = set(labels)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Noise points
            col = 'gray'
            marker = 'x'
            alpha = 0.3
            label = 'Noise'
        else:
            marker = 'o'
            alpha = 0.7
            label = f'Cluster {k}'
        
        class_member_mask = (labels == k)
        xy = X[class_member_mask]
        ax.scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, 
                  s=50, alpha=alpha, edgecolors='k', linewidth=0.5, label=label)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Feature 1', fontsize=12)
    ax.set_ylabel('Feature 2', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    return ax

def print_clustering_stats(labels, elapsed_time=None):
    """Print clustering statistics."""
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    print(f"Number of clusters: {n_clusters}")
    print(f"Number of noise points: {n_noise}")
    if elapsed_time:
        print(f"Clustering time: {elapsed_time:.4f} seconds")
    print()

def make_varied_density_blobs(n_samples=300, random_state=42):
    """Create dataset with clusters of varying densities."""
    np.random.seed(random_state)
    
    # Dense cluster
    dense = np.random.randn(n_samples // 3, 2) * 0.3 + np.array([0, 0])
    
    # Medium density cluster
    medium = np.random.randn(n_samples // 3, 2) * 0.8 + np.array([5, 0])
    
    # Sparse cluster
    sparse = np.random.randn(n_samples // 3, 2) * 1.5 + np.array([0, 5])
    
    X = np.vstack([dense, medium, sparse])
    y = np.hstack([np.zeros(n_samples // 3), 
                   np.ones(n_samples // 3), 
                   np.ones(n_samples // 3) * 2])
    
    return X, y.astype(int)

## 1. Varying Density Clusters

HDBSCAN's key advantage: handling clusters with different densities.

In [ ]:
# Generate varying density data
X_varied, y_varied = make_varied_density_blobs(n_samples=300)

# Run HDBSCAN
print("HDBSCAN on Varying Density Clusters")
print("=" * 50)
hdbscan = HDBSCAN(min_cluster_size=15, min_samples=5)

start_time = time.time()
labels = hdbscan.fit_predict(X_varied)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original data
ax1.scatter(X_varied[:, 0], X_varied[:, 1], c=y_varied, cmap='viridis', 
           s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
ax1.set_title('Original Data (Varying Densities)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.grid(True, alpha=0.3)

# HDBSCAN results
plot_clusters(X_varied, labels, 
             'HDBSCAN Results (min_cluster_size=15, min_samples=5)', ax=ax2)

plt.tight_layout()
plt.show()

## 2. Moons Dataset

Testing HDBSCAN on non-convex clusters.

In [ ]:
# Generate moons dataset
X_moons, y_moons = datasets.make_moons(n_samples=300, noise=0.05, random_state=42)

# Run HDBSCAN
print("HDBSCAN on Moons Dataset")
print("=" * 50)
hdbscan = HDBSCAN(min_cluster_size=15, min_samples=5)

start_time = time.time()
labels = hdbscan.fit_predict(X_moons)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original data
ax1.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', 
           s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
ax1.set_title('Original Data (Ground Truth)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.grid(True, alpha=0.3)

# HDBSCAN results
plot_clusters(X_moons, labels, 
             'HDBSCAN Results (min_cluster_size=15, min_samples=5)', ax=ax2)

plt.tight_layout()
plt.show()

## 3. Noisy Blobs

Testing HDBSCAN's robustness to noise and outliers.

In [ ]:
# Generate data with noise
X_blobs, y_blobs = datasets.make_blobs(n_samples=250, centers=4, 
                                        cluster_std=0.6, random_state=42)

# Add random outliers
outliers = np.random.uniform(low=-10, high=10, size=(50, 2))
X_noisy = np.vstack([X_blobs, outliers])

# Run HDBSCAN
print("HDBSCAN on Noisy Blobs")
print("=" * 50)
hdbscan = HDBSCAN(min_cluster_size=15, min_samples=5)

start_time = time.time()
labels = hdbscan.fit_predict(X_noisy)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
plot_clusters(X_noisy, labels, 
             'HDBSCAN on Noisy Data (min_cluster_size=15, min_samples=5)', ax=ax)
plt.show()

## 4. Iris Dataset

Testing HDBSCAN on real-world data.

In [ ]:
# Load Iris dataset
iris = datasets.load_iris()
X_iris = iris.data[:, :2]  # Use first 2 features for visualization
y_iris = iris.target

# Standardize features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Run HDBSCAN
print("HDBSCAN on Iris Dataset (first 2 features)")
print("=" * 50)
hdbscan = HDBSCAN(min_cluster_size=10, min_samples=5)

start_time = time.time()
labels = hdbscan.fit_predict(X_iris_scaled)
elapsed = time.time() - start_time

print_clustering_stats(labels, elapsed)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Original data
ax1.scatter(X_iris_scaled[:, 0], X_iris_scaled[:, 1], c=y_iris, cmap='viridis', 
           s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
ax1.set_title('Original Data (Ground Truth)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sepal Length (scaled)', fontsize=12)
ax1.set_ylabel('Sepal Width (scaled)', fontsize=12)
ax1.grid(True, alpha=0.3)

# HDBSCAN results
plot_clusters(X_iris_scaled, labels, 
             'HDBSCAN Results (min_cluster_size=10, min_samples=5)', ax=ax2)
ax2.set_xlabel('Sepal Length (scaled)', fontsize=12)
ax2.set_ylabel('Sepal Width (scaled)', fontsize=12)

plt.tight_layout()
plt.show()

## 5. Comparison with DBSCAN

Comparing HDBSCAN and DBSCAN on the same dataset.

In [ ]:
from ghdbscan import DBSCAN

# Use varying density data
X_comp, y_comp = make_varied_density_blobs(n_samples=300)

# Run DBSCAN
print("DBSCAN vs HDBSCAN Comparison")
print("=" * 50)
print("\nDBSCAN:")
dbscan = DBSCAN(eps=0.8, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_comp)
print_clustering_stats(labels_dbscan)

print("HDBSCAN:")
hdbscan = HDBSCAN(min_cluster_size=15, min_samples=5)
labels_hdbscan = hdbscan.fit_predict(X_comp)
print_clustering_stats(labels_hdbscan)

# Visualize comparison
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

# Original data
ax1.scatter(X_comp[:, 0], X_comp[:, 1], c=y_comp, cmap='viridis', 
           s=50, alpha=0.7, edgecolors='k', linewidth=0.5)
ax1.set_title('Original Data\n(Varying Densities)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Feature 1', fontsize=12)
ax1.set_ylabel('Feature 2', fontsize=12)
ax1.grid(True, alpha=0.3)

# DBSCAN results
plot_clusters(X_comp, labels_dbscan, 'DBSCAN\n(eps=0.8, min_samples=5)', ax=ax2)

# HDBSCAN results
plot_clusters(X_comp, labels_hdbscan, 
             'HDBSCAN\n(min_cluster_size=15, min_samples=5)', ax=ax3)

plt.tight_layout()
plt.show()

print("\n📊 Observation:")
print("HDBSCAN better handles varying density clusters without manual eps tuning.")

## 6. Parameter Sensitivity

Exploring how `min_cluster_size` affects results.

In [ ]:
# Use moons dataset
X_test, _ = datasets.make_moons(n_samples=300, noise=0.05, random_state=42)

min_cluster_sizes = [5, 10, 20, 30]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, min_size in enumerate(min_cluster_sizes):
    hdbscan = HDBSCAN(min_cluster_size=min_size, min_samples=5)
    labels = hdbscan.fit_predict(X_test)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    title = f'min_cluster_size={min_size}, min_samples=5\n({n_clusters} clusters, {n_noise} noise points)'
    plot_clusters(X_test, labels, title, ax=axes[idx])

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the HDBSCAN clustering algorithm on various datasets:

1. **Varying Density Clusters**: Successfully identified clusters with different densities
2. **Moons**: Handled non-convex cluster shapes
3. **Noisy Blobs**: Effectively separated outliers from clusters
4. **Iris**: Clustered real-world botanical data
5. **DBSCAN Comparison**: Showed advantages over standard DBSCAN
6. **Parameter Sensitivity**: Demonstrated effect of `min_cluster_size`

### Key Advantages of HDBSCAN:

- ✅ **No eps parameter**: Automatically adapts to varying densities
- ✅ **Hierarchical structure**: Provides cluster hierarchy at multiple scales
- ✅ **Robust to noise**: Effectively identifies and separates outliers
- ✅ **Varying densities**: Handles clusters with different densities
- ✅ **Stability-based**: Selects most stable clusters from hierarchy

### When to use HDBSCAN over DBSCAN:

- Clusters have varying densities
- You want to avoid manual parameter tuning
- You need hierarchical cluster structure
- You want cluster membership probabilities